# 102 Workshop — Day 1 Tasks

This notebook covers **Task 1a** (Python functions) and **Task 1b** (lists and dictionaries).

Run each cell with `Shift + Enter`. Fill in the `# TODO` sections.

## Task 1a — Python Functions

Create a function for each of the following. Test your work in the cells below.

### 1. Square of a number
Write a function `square(n)` that returns the square of `n`.

In [ ]:
def square(n):
    # TODO: return the square of n
    pass


# Test
print(square(5))   # Expected: 25
print(square(-3))  # Expected: 9


### 2. Maximum of three numbers
Write a function `max_of_three(a, b, c)` that returns the largest of the three.

In [ ]:
def max_of_three(a, b, c):
    # TODO: return the largest of a, b, c
    pass


# Test
print(max_of_three(1, 2, 3))     # Expected: 3
print(max_of_three(10, 5, 7))    # Expected: 10
print(max_of_three(-1, -5, -2))  # Expected: -1


### 3. Sum all numbers in a list
Write a function `sum_list(numbers)` that returns the sum of every number in the list.

In [ ]:
def sum_list(numbers):
    # TODO: return the sum of all numbers in the list
    # Hint: you can use a for loop, or Python's built-in sum()
    pass


# Test
print(sum_list([1, 2, 3, 4, 5]))  # Expected: 15
print(sum_list([10, -5, 3]))      # Expected: 8
print(sum_list([]))               # Expected: 0


### 4. Check if a number is within a range
Write a function `in_range(n, low, high)` that returns `True` if `low <= n <= high`, otherwise `False`.

In [ ]:
def in_range(n, low, high):
    # TODO: return True if n is between low and high (inclusive), else False
    pass


# Test
print(in_range(5, 1, 10))    # Expected: True
print(in_range(15, 1, 10))   # Expected: False
print(in_range(10, 1, 10))   # Expected: True (inclusive)


---

## Task 1b — Lists and Dictionaries

In FastAPI, data almost always moves around as **lists of dictionaries** — that's exactly what JSON looks like when it arrives from a client or gets saved to a file. Before we touch FastAPI, let's get comfortable with the operations you'll use over and over:

- creating and accessing dictionaries
- looping over a list of dictionaries
- finding an item by its `id`
- filtering a list
- adding, updating, and removing items

These are the exact patterns used inside every CRUD endpoint you'll build today.

### 1. Dictionaries — the basics

A dictionary stores **key → value** pairs. In an API, each "task" or "user" or "product" is a dictionary.

In [ ]:
task = {
    "id": 1,
    "description": "Buy milk",
    "completed": False,
}

# Access a value by key
print(task["description"])   # Buy milk

# .get() is safer — it returns None (or a default) if the key is missing
print(task.get("description"))       # Buy milk
print(task.get("priority"))          # None
print(task.get("priority", "low"))   # low  (default when key not found)


**Why `.get()` matters for APIs:** when a request comes in, some fields might be missing. Using `task["priority"]` on a missing key raises an error and crashes your endpoint. Using `task.get("priority")` returns `None` safely.

In [ ]:
# TODO: create a dictionary called `me` with keys: name, age, is_student
me = {
    # ...
}

# TODO: print just your name using .get()
# print(...)


### 2. Updating and adding fields

Dictionaries are mutable — you can change existing values and add new keys after creation.

In [ ]:
task = {"id": 1, "description": "Buy milk", "completed": False}

# Update an existing field
task["completed"] = True

# Add a new field that wasn't there before
task["category"] = "shopping"

print(task)
# {'id': 1, 'description': 'Buy milk', 'completed': True, 'category': 'shopping'}


In [ ]:
# TODO: starting from this task, mark it completed and add a priority of "high"
task = {"id": 2, "description": "Walk dog", "completed": False}

# Your code here


print(task)


### 3. Lists of dictionaries — the API data shape

Your Task Manager API will store all tasks as a list of dictionaries. This is how the data looks in `data.json` too.

In [ ]:
tasks = [
    {"id": 1, "description": "Buy milk", "completed": False},
    {"id": 2, "description": "Walk dog", "completed": True},
    {"id": 3, "description": "Read book", "completed": False},
]

print(f"Total tasks: {len(tasks)}")
print(f"First task: {tasks[0]}")
print(f"Description of first task: {tasks[0]['description']}")


### 4. Looping over a list of dictionaries

This is the single most common pattern in your CRUD endpoints. Every `GET /tasks/{id}`, `PUT`, and `DELETE` does a loop like this.

In [ ]:
# Print the description of every task
for task in tasks:
    print(task["description"])


In [ ]:
# TODO: loop over `tasks` and print ONLY the descriptions of completed tasks
# Hint: use an `if` inside the loop

for task in tasks:
    # ...
    pass


### 5. Finding an item by ID

This is exactly what `GET /tasks/{task_id}` does under the hood.

In [ ]:
def find_task(task_id):
    for task in tasks:
        if task["id"] == task_id:
            return task
    return None   # not found


print(find_task(2))    # {'id': 2, 'description': 'Walk dog', 'completed': True}
print(find_task(999))  # None


In [ ]:
# TODO: write a function find_by_description(keyword) that returns the FIRST
# task whose description contains the keyword (case-insensitive).
# Return None if no task matches.
#
# Hint: use the `in` operator on lowercased strings
#   "milk" in "Buy milk".lower()  -> True

def find_by_description(keyword):
    # ...
    pass


print(find_by_description("milk"))   # Expected: the "Buy milk" task
print(find_by_description("MILK"))   # Expected: same task (case-insensitive)
print(find_by_description("xyz"))    # Expected: None


### 6. Filtering — returning a subset

Where `find_task` returns one item, **filtering** returns many. Your `GET /tasks?completed=true` endpoint is just a filter.

In [ ]:
# The verbose way: build a new list with a for loop
completed_tasks = []
for task in tasks:
    if task["completed"]:
        completed_tasks.append(task)

print(completed_tasks)


In [ ]:
# The Pythonic way: a list comprehension. Same result, one line.
completed_tasks = [t for t in tasks if t["completed"]]

print(completed_tasks)


Both work. The list comprehension is what you'll see in most FastAPI code. Read it as: *"t for each t in tasks, if t is completed"*.

In [ ]:
# TODO: write a list comprehension that returns all INCOMPLETE tasks
# incomplete = ...
# print(incomplete)


### 7. Adding a new item (the POST pattern)

When a client sends `POST /tasks`, you assign an ID and append to the list. Here's the pattern:

In [ ]:
# Find the next available ID. `default=0` handles the case when the list is empty.
new_id = max([t["id"] for t in tasks], default=0) + 1

new_task = {
    "id": new_id,
    "description": "Do laundry",
    "completed": False,
}
tasks.append(new_task)

print(f"Added task with id {new_id}")
print(tasks)


#### The `**` unpack trick

You'll see this exact line in your FastAPI solution:

```python
new_task = {"id": new_id, **task.model_dump()}
```

The `**` "spreads" a dictionary's keys into another dictionary — a clean way to merge data.

In [ ]:
incoming = {"description": "Call mum", "completed": False}
new_id = 99

# Merge: add the id on top of whatever came in
new_task = {"id": new_id, **incoming}
print(new_task)
# {'id': 99, 'description': 'Call mum', 'completed': False}


### 8. Updating an item (the PUT pattern)

To update, loop until you find the item, then replace it.

In [ ]:
def update_task(task_id, new_description):
    for i, task in enumerate(tasks):
        if task["id"] == task_id:
            tasks[i]["description"] = new_description
            return tasks[i]
    return None


updated = update_task(1, "Buy oat milk")
print(updated)


`enumerate()` gives you both the **index** (`i`) and the **item** (`task`) on each loop — useful when you need to modify the list in place.

### 9. Removing an item (the DELETE pattern)

In [ ]:
def delete_task(task_id):
    for i, task in enumerate(tasks):
        if task["id"] == task_id:
            return tasks.pop(i)   # pop removes AND returns the item
    return None


print(delete_task(3))   # returns the removed task
print(tasks)            # task 3 is gone


### 10. Putting it all together

You now know every data-manipulation pattern needed for a CRUD API. To recap, each HTTP method maps to one of these:

| Operation | HTTP | Pattern |
|---|---|---|
| List all | `GET /tasks` | return the list |
| Filter | `GET /tasks?completed=true` | list comprehension |
| Get one | `GET /tasks/{id}` | loop + `if id ==` + return |
| Create | `POST /tasks` | compute new id, append |
| Update | `PUT /tasks/{id}` | loop with `enumerate`, replace |
| Delete | `DELETE /tasks/{id}` | loop with `enumerate`, `.pop(i)` |

### Challenge (optional)

Using only the patterns above, write a function `summary(tasks)` that returns a dictionary like:

```python
{"total": 4, "completed": 1, "pending": 3}
```

In [ ]:
def summary(tasks):
    # TODO: return a dict with keys "total", "completed", "pending"
    pass


# Test with a fresh list
sample = [
    {"id": 1, "description": "A", "completed": True},
    {"id": 2, "description": "B", "completed": False},
    {"id": 3, "description": "C", "completed": False},
    {"id": 4, "description": "D", "completed": False},
]
print(summary(sample))   # Expected: {'total': 4, 'completed': 1, 'pending': 3}
